# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and processing the FAIR<sup>2</sup> dataset ("Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells") using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

-----
### Dataset Source
The dataset is described by a [Croissant schema](https://www.mlcommons.org/croissant/) and is referenced by the schema URL below. All fields, record sets, and columns are uniquely referenced by their `@id`, for reliability and clarity in multi-table FAIR data workflows.

In [ ]:
# Ensure mlcroissant is installed (uncomment if necessary)
!pip install --quiet mlcroissant

## 1. Data Loading

We load the Croissant schema metadata and inspect top-level information about the dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {getattr(metadata, 'name', None)}\n\nDescription: {getattr(metadata, 'description', None)}")

## 2. Data Overview

Discover available record sets (`@id`), and for each record set, enumerate their fields and corresponding column `@id`s. This is necessary for targeted field selection and referencing during EDA and downstream workflows.

In [ ]:
# List all available record sets and their fields, using entity @id
from pprint import pprint

def get_record_sets(dataset):
    """Retrieve the list of record set objects with their @id."""
    # dataset.record_sets is a list of class RecordSet
    # Each has an .id attribute (the @id)
    return dataset.record_sets

def get_fields_for_record_set(record_set):
    """Return a list of (field @id, column @id, dataType) tuples for a record set."""
    # Each field in record_set.fields is a Field object with .id and .column (.id), .data_type.
    fields = []
    for field in record_set.fields:
        col_id = getattr(field, 'column', None)
        if col_id is not None:
            if hasattr(col_id, 'id'):
                col_id_value = col_id.id
            else:
                col_id_value = col_id # Might already be a string
        else:
            col_id_value = None
        fields.append((field.id, col_id_value, getattr(field, 'data_type', None)))
    return fields

# Print overview
print("Available record sets in dataset schema:")
record_sets = get_record_sets(dataset)
recset_ids = []
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id} (name: {getattr(rs, 'name', None)})")
    recset_ids.append(rs.id)
    fields = get_fields_for_record_set(rs)
    print("  Fields:")
    for fid, colid, dtype in fields:
        print(f"    - Field @id: {fid} | column @id: {colid} | dataType: {dtype}")

## 3. Data Extraction

Load data for each record set into a DataFrame using their `@id`. You can select specific record sets and reference their fields by `@id` in analysis and visualization.

In [ ]:
dataframes = {}

# We'll extract a sample of records from each record set
for record_set_id in recset_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if len(records):
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet {record_set_id}.")
            print(f"Columns (by field @id): {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records found for RecordSet {record_set_id}.")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# For demonstration, pick the first available record set id as the main analysis target
main_record_set_id = recset_ids[0] if recset_ids else None

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps such as filtering, normalization, and grouping to numeric fields. Demonstrate on the main record set, referencing columns by their field `@id`.

*Steps:*
- Select a numeric field by `@id` from the overview above.
- Filter records, normalize a numeric field, display results.
- Optionally group by a categorical field if present.

In [ ]:
# You can specify the field @id for numeric analysis
# For demonstration, we attempt to auto-select the first numeric field
# Otherwise, set a field @id manually if known
from pandas.api.types import is_numeric_dtype

df = dataframes[main_record_set_id]

# Find a numeric field from the dataframe (by field @id)
numeric_field = None
for col in df.columns:
    if is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is None:
    print("No numeric field found. Please assign 'numeric_field' to a numeric field @id from the overview above.")
else:
    print(f"Using numeric field (field @id): {numeric_field}")
    # Example threshold
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to group by a likely categorical field (by @id)
    group_field = None
    for col in df.columns:
        # Simple heuristic: try columns with object type, not numeric, not obviously id
        if df[col].dtype == 'object' and 'id' not in col.lower() and col != numeric_field:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
        display(grouped_df.head())

## 5. Visualization

Visualize distributions or relationships between fields using matplotlib or pandas built-in plotting. All plots reference fields by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field:
    plt.figure(figsize=(7, 4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of field '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 4))
        grouped_df.set_index(group_field)[numeric_field].plot(kind='bar')
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion

In this notebook, you've successfully:
- Loaded and explored the FAIR<sup>2</sup> dataset using the `mlcroissant` library.
- Enumerated entities (record sets, fields) by their unique `@id`, supporting robust FAIR workflows.
- Extracted data frames and performed EDA on numeric and categorical features.
- Produced basic visualizations referencing `@id` fields.  

**Next steps:**
- Explore additional record sets and utilize their fields by `@id` for downstream analysis.
- Integrate advanced processing or ML methods, referencing schema metadata for full automation and reproducibility.
- See [mlcroissant documentation](https://croissant.mlcommons.org/) for more advanced API features!